# VVC_RL D-Series — controlled preflight and training

Default **D4**: D3 state/Actor/reward with analytic critic budget decomposition. Preflight only by default; training and optional fixed diagnostics remain disabled.

Functional source commit: `c79f10308b2aded88ea8c48c88a75952a13e7f68`. GPU **T4 x2**, Internet ON. Same locked mini16 Train / 4 Validation. No Test use. Fresh D4 gate/checkpoints only; no D3/B/Combined resume.

Inputs may be intact SAFE ZIPs or Kaggle-expanded `VVC_RL_payload.bin` + `payload_sha256.txt`; toolchain ZIP or expanded manifest/executable directory. Do not manually expand the inner payload.bin.

In [ ]:
from pathlib import Path
import sys, os, json, time, subprocess, shutil
SOURCE_COMMIT = 'c79f10308b2aded88ea8c48c88a75952a13e7f68'
EXPECTED_SOURCE_SHA256 = 'e44a51fc01adcb986c53614fd596405328cbc40c080799d7f665129c5775577f'
REPOSITORY = 'https://github.com/youngan2810/VVC_RL.git'
VARIANTS = {
    'D0': ('encoder-feedback-v1','task-budget-v3-progressive-soft-iou'),
    'D1': ('d1-decoupled-v1','task-budget-v3-progressive-soft-iou'),
    'D1i': ('d1-isolated-v1','task-budget-v3-progressive-soft-iou'),
    'D2': ('d2-region-v1','task-budget-v3-progressive-soft-iou'),
    'D2-neutral': ('d2-region-neutral-v1','task-budget-v3-progressive-soft-iou'),
    'D3': ('d3-online-budget-v1','task-budget-v4-online-soft-iou'),
    'D4': ('d4-budget-value-v1','task-budget-v4-online-soft-iou'),
}
VARIANT = 'D4'
SEED = 20261120
FRAMES = 256
VAL_EVERY = 5
SESSION_HOURS = 11.0
RUN_MINI_TRAINING = False
ADDITIONAL_UPDATES = 10
RESUME_EXISTING = False
RESUME_PACKAGE = ''  # absolute path to same-variant, same-source SAFE ZIP
RUN_FIXED_DIAGNOSTICS = False  # separate optional 48-encode validation after training
assert VARIANT in VARIANTS and 0 < SESSION_HOURS <= 11.75
assert type(ADDITIONAL_UPDATES) is int and 0 <= ADDITIONAL_UPDATES <= 40
STATE_VERSION, REWARD_VERSION = VARIANTS[VARIANT]
WORK=Path('/kaggle/working'); INPUT=Path('/kaggle/input')
assert WORK.is_dir() and INPUT.is_dir(), 'Kaggle BDD100K input required'
BASE=WORK/('VVC_RL_D_'+SOURCE_COMMIT[:8]); REPO=BASE/'source'; TOOLS=BASE/'tools'
TAG=f'{VARIANT}_seed{SEED}'; TRAIN=WORK/f'vvc_rl_{TAG}_{SOURCE_COMMIT[:8]}'
PREFLIGHT=WORK/f'preflight_{TAG}_{time.time_ns()}'
LOGS=WORK/f'logs_{TAG}_{time.time_ns()}'; LOGS.mkdir(parents=True)
STARTED=time.monotonic()
os.environ['YOLOv5_AUTOINSTALL']='false'
print(VARIANT, STATE_VERSION, REWARD_VERSION, TRAIN, PREFLIGHT)

In [ ]:
def live(args,name,cwd=None):
    args=list(map(str,args)); log=LOGS/f'{name}.log'
    print('$',*args,flush=True)
    with log.open('w',encoding='utf-8') as out:
        p=subprocess.Popen(args,cwd=cwd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,errors='replace')
        try:
            for line in p.stdout:
                out.write(line);out.flush();print(line,end='',flush=True)
            if p.wait():raise RuntimeError(f'{name} failed; see {log}')
        finally:
            if p.poll() is None:p.terminate();p.wait()
    return log

def common_args():
    return ['--toolchain',TOOLS/'TOOLCHAIN.json','--tools',TOOLS,'--input',INPUT,
        '--frames',FRAMES,'--state-version',STATE_VERSION,'--reward-version',REWARD_VERSION,
        '--rate-tolerance','0.02','--gamma','1.0','--gae-lambda','0.95','--train-schedule','mixed','--seed',SEED]

def verify_source():
    script="""import sys,subprocess
from vvcrl.repository import verify_checkout
from vvcrl.config import fingerprint
assert subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip()==sys.argv[1]
print('TRACKED_RAW_BYTES_PASS',verify_checkout('.'))
# No untracked functional input may silently change the source fingerprint.
assert not subprocess.check_output(['git','ls-files','--others','--exclude-standard','--','vvcrl','native','vendor','configs','scripts','tests'],text=True).strip()
assert fingerprint('.')==sys.argv[2], 'Functional source differs from this notebook'
print('FUNCTIONAL_SOURCE_SHA256',fingerprint('.'))
"""
    live([sys.executable,'-c',script,SOURCE_COMMIT,EXPECTED_SOURCE_SHA256],'source_verify',REPO)

def gate_check():
    script="""from pathlib import Path
import json,sys
from vvcrl.gates import binding,require_gate
from vvcrl.config import Profile,PPOConfig,RewardConfig
p,t,sv,rv,frames=sys.argv[1:]
b=binding(Path.cwd(),json.loads(Path(t).read_text()),Profile(frames=int(frames)),2,RewardConfig(rv,.02),PPOConfig(),'mixed',sv)
print('GATE_RECHECK_PASS',require_gate(p,b)['status'])
"""
    live([sys.executable,'-c',script,PREFLIGHT,TOOLS/'TOOLCHAIN.json',STATE_VERSION,REWARD_VERSION,FRAMES],'gate_recheck',REPO)

## Setup and exact source binding
Do not edit source or bypass source, binary or environment gates. The locked dataset contains 16 train and 4 validation clips for mini training. No Test access.

In [ ]:
BASE.mkdir(parents=True,exist_ok=True)
if not REPO.exists():live(['git','clone','--branch','research/d-series-adaptive-policy',REPOSITORY,REPO],'clone')
else:
    assert (REPO/'.git').is_dir()
    live([sys.executable,'-c',"from vvcrl.repository import verify_checkout; print('TRACKED_RAW_BYTES_PASS',verify_checkout('.'))"],'existing_source_verify',REPO)
live(['git','-C',REPO,'fetch','--no-tags','origin',SOURCE_COMMIT],'fetch')
live(['git','-C',REPO,'checkout','--detach',SOURCE_COMMIT],'checkout')
verify_source()
live([sys.executable,'-c','import torch; print(torch.__version__,torch.cuda.device_count()); assert torch.cuda.is_available() and torch.cuda.device_count()>=2'], 'gpu_check',REPO)
live([sys.executable,'-c',"from vvcrl.locked.data import VideoCatalog; import sys; c=VideoCatalog(sys.argv[1],mode='mini',seed=20261111); c.check_available(('train','val')); print('DATASET_PASS',len(c.selected['train']),len(c.selected['val']))",INPUT],'dataset_check',REPO)
print('FREE_GB',shutil.disk_usage(WORK).free/1e9)
live([sys.executable,'-u',REPO/'scripts/setup_kaggle.py','--tools',TOOLS,'--jobs',2],'setup',REPO)
verify_source()

## Full preflight
One disposable real-data PPO update, independent decoding, CPU/native contracts, two GPU workers, checkpoint reload and real validation. Do not use the mechanical checkpoint for training. Gate failure blocks training.

In [ ]:
if not PREFLIGHT.exists():
    live([sys.executable,'-u','-m','vvcrl','preflight','--output',PREFLIGHT,*common_args()],'preflight',REPO)
gate_check()
gate=json.loads((PREFLIGHT/'PREFLIGHT_GATE.json').read_text())
assert gate['status']=='PASS_READY_FOR_MINI'
print(json.dumps(gate['checks'],indent=2))
stats=json.loads((PREFLIGHT/'PPO_MECHANICAL_UPDATE.json').read_text())['stats']
print('PPO_MECHANICS',json.dumps(stats,indent=2))
summary=json.loads((PREFLIGHT/'REAL_DATA_SUMMARY.json').read_text())
assert summary['test_used'] is False and summary['policy_changed_on_validation'] is False
for split in ('train','validation'):
    for r in summary[split]:
        assert r['reward_identity_error']<1e-10
        print(split,r['rate'],r['actual_kbps'],r['task_retention_f1_v2'],r['reward_identity_error'])
print('PREFLIGHT_SAFE',WORK/f'VVC_RL_PREFLIGHT_{PREFLIGHT.name}_SAFE.zip')

## Controlled training
After reviewing PASS, set RUN_MINI_TRAINING=True in the first code cell (or set it in a new cell in the same session) and run the next cell. Start with 10 updates, validation every 5. D1 tests representation isolation alone; D1i/D2/D3 are subsequent controlled variants. Resume only matching latest SAFE from this exact source/variant/seed/environment/binaries. Do not resume historical Combined. At 40 updates with pending validation, use ADDITIONAL_UPDATES=0. No guarantee all updates finish in one Kaggle session.

In [ ]:
if RUN_MINI_TRAINING:
    verify_source();gate_check()
    if RESUME_PACKAGE:
        assert RESUME_EXISTING
        package=Path(RESUME_PACKAGE)
        assert package.is_file() and package.suffix=='.zip' and '_FAILED_' not in package.name
        live([sys.executable,'-u','-m','vvcrl','import','--output',TRAIN,'--package',package],'resume_import',REPO)
    resume=[]
    if RESUME_EXISTING:
        latest=TRAIN/'checkpoints/latest.pt';assert latest.is_file()
        progress=json.loads((TRAIN/'PROGRESS.json').read_text())
        assert progress['iteration']+ADDITIONAL_UPDATES<=40
        assert ADDITIONAL_UPDATES>0 or progress['validation_pending']
        resume=['--resume',latest]
    else:
        assert ADDITIONAL_UPDATES>0 and (not TRAIN.exists() or not any(TRAIN.iterdir()))
    remaining=SESSION_HOURS-(time.monotonic()-STARTED)/3600
    assert remaining>0.5,'Insufficient session time'
    live([sys.executable,'-u','-m','vvcrl','train-mini','--output',TRAIN,'--preflight',PREFLIGHT,
        '--updates',ADDITIONAL_UPDATES,'--val-every',VAL_EVERY,'--hours',remaining,*common_args(),'--start-mini-training',*resume],'training',REPO)
else:print('TRAINING_DISABLED: review preflight before enabling')

## Optional compact fixed-policy validation
Disabled by default. Only after a completed validation checkpoint, use the same toolchain and checkpoint prices. 4 videos × 3 rates × (PPO + 3 fixed pairs), plus anchors. This is development only, not independent test performance. Run as a separate session if time is limited.

In [ ]:
if RUN_FIXED_DIAGNOSTICS:
    verify_source();gate_check()
    checkpoint=TRAIN/'checkpoints/best_validation.pt'
    assert checkpoint.is_file(),'No eligible F1 checkpoint; do not silently select another'
    out=WORK/f'fixed_diagnostics_{TAG}_{time.time_ns()}'
    live([sys.executable,'-u','-m','vvcrl','diagnose','--output',out,'--preflight',PREFLIGHT,
        '--resume',checkpoint,'--diagnostic-mode','compact',*common_args()],'fixed_diagnostics',REPO)
else:print('FIXED_DIAGNOSTICS_DISABLED')

In [ ]:
for p in sorted(WORK.glob('VVC_RL_*_SAFE.zip')):
    print('SAFE',p,'MiB',round(p.stat().st_size/2**20,1))
if (TRAIN/'PROGRESS.json').is_file():print('PROGRESS',(TRAIN/'PROGRESS.json').read_text())
for p in sorted((TRAIN/'validations').glob('*.json')):
    r=json.loads(p.read_text());print('VAL',p.stem,r['complete'],r['feasible'],r['macro_delta_quality'],r['macro_delta_utility'])
print('LOGS',LOGS)